# Best Model — RoBERTa-base with Data Augmentation

- **Model**: `roberta-base` via `AutoModelForSequenceClassification`
- **Training data**: official train split + `augmented_train_data.csv` (5558 synthetic PCL samples)
- **Optimiser**: AdamW (`adamw_torch`) with cosine schedule and 10% warmup
- **Class imbalance**: class-weighted CrossEntropyLoss
- **Best model**: saved by highest dev macro-F1 during training
- **Outputs**: `dev.txt` (2094 lines) and `test.txt` (3832 lines) — one `0`/`1` prediction per line

In [ ]:
!pip install contractions python-dotenv huggingface_hub

In [ ]:
import os
import re
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
)
from sklearn.utils.class_weight import compute_class_weight

import matplotlib.pyplot as plt
import seaborn as sns
import contractions

from dotenv import load_dotenv
from huggingface_hub import login

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
load_dotenv()
hf_token = os.getenv('HF_TOKEN')
if hf_token:
    login(token=hf_token)
    print('HF token loaded.')
else:
    print('HF_TOKEN not found; proceeding without login.')

In [ ]:
# ============================================================
# Configuration
# ============================================================
MODEL_NAME  = 'roberta-base'
RUN_NAME    = 'checkpoints/roberta_best'
MAX_LENGTH  = 256
NUM_EPOCHS  = 5

DATA_ROOT      = '.'
TSV_PATH       = os.path.join(DATA_ROOT, 'dontpatronizeme_pcl.tsv')
TRAIN_IDS_PATH = os.path.join(DATA_ROOT, 'train', 'train_semeval_parids-labels.csv')
DEV_IDS_PATH   = os.path.join(DATA_ROOT, 'train', 'dev_semeval_parids-labels.csv')
TEST_PATH      = os.path.join(DATA_ROOT, 'test', 'task4_test.tsv')
AUG_DATA_PATH  = os.path.join(DATA_ROOT, 'data_augmentation', 'augmented_train_data.csv')

DEV_TXT_PATH  = 'dev.txt'
TEST_TXT_PATH = 'test.txt'

os.makedirs(RUN_NAME, exist_ok=True)
print(f'Checkpoint dir: {RUN_NAME}')

In [ ]:
# ============================================================
# Helper functions
# ============================================================
def load_task1(tsv_path: str) -> pd.DataFrame:
    """Load labelled PCL dataset; binarise labels (0/1 -> 0, 2/3/4 -> 1)."""
    rows = []
    with open(tsv_path, encoding='utf-8') as f:
        for line in f.readlines()[4:]:
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 6:
                continue
            orig_label = parts[-1]
            rows.append({
                'par_id':  str(parts[0]),
                'art_id':  parts[1],
                'keyword': parts[2],
                'country': parts[3],
                'text':    parts[4],
                'label':   0 if orig_label in {'0', '1'} else 1,
            })
    return pd.DataFrame(rows)


def load_test_tsv(test_path: str) -> pd.DataFrame:
    """Load unlabelled test set (tab-separated, no header)."""
    rows = []
    with open(test_path, encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 5:
                continue
            rows.append({
                'par_id':  parts[0],
                'art_id':  parts[1],
                'keyword': parts[2],
                'country': parts[3],
                'text':    parts[4],
            })
    return pd.DataFrame(rows)


def preprocess_text(text: str) -> str:
    text = str(text)
    text = contractions.fix(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def make_model_text(frame: pd.DataFrame) -> pd.Series:
    """Concatenate keyword, country, and cleaned text with </s> separators."""
    kw  = frame['keyword'].fillna('').astype(str).str.strip()
    cc  = frame['country'].fillna('').astype(str).str.strip()
    txt = frame['clean_text'].fillna('').astype(str).str.strip()
    return kw + ' </s> ' + cc + ' </s> ' + txt

In [ ]:
# ============================================================
# Load labelled dataset + preprocess
# ============================================================
df = load_task1(TSV_PATH)
df['clean_text'] = df['text'].apply(preprocess_text)
df['model_text'] = make_model_text(df)

print(f'Full labelled dataset: {len(df):,} rows')
print(df['label'].value_counts().rename({0: 'No-PCL', 1: 'PCL'}))

In [ ]:
# ============================================================
# Official train / dev split
# ============================================================
train_ids_df = pd.read_csv(TRAIN_IDS_PATH, dtype={'par_id': str})
dev_ids_df   = pd.read_csv(DEV_IDS_PATH,   dtype={'par_id': str})

train_par_ids = set(train_ids_df['par_id'].astype(str))
dev_par_ids   = set(dev_ids_df['par_id'].astype(str))

train_df = df[df['par_id'].isin(train_par_ids)].copy().reset_index(drop=True)

# Dev set ordered exactly as in the official split file (required for dev.txt)
dev_df = (
    dev_ids_df[['par_id']]
    .merge(df[df['par_id'].isin(dev_par_ids)], on='par_id', how='left')
    .reset_index(drop=True)
)

# Any leftover samples not in either split go to train
leftover = df[~df['par_id'].isin(train_par_ids | dev_par_ids)]
if len(leftover):
    train_df = pd.concat([train_df, leftover], ignore_index=True)
    print(f'Appended {len(leftover):,} leftover samples to train.')

print(f'Train (before augmentation): {len(train_df):,} | PCL={int((train_df.label==1).sum()):,} | No-PCL={int((train_df.label==0).sum()):,}')
print(f'Dev:                         {len(dev_df):,}  | PCL={int((dev_df.label==1).sum()):,}  | No-PCL={int((dev_df.label==0).sum()):,}')

In [ ]:
# ============================================================
# Load augmented training data and concatenate
# ============================================================
aug_df = pd.read_csv(AUG_DATA_PATH, dtype={'par_id': str})

# Drop any accidental duplicate-header rows
aug_df = aug_df[aug_df['label'].astype(str) != 'label'].copy()
aug_df['label'] = aug_df['label'].astype(int)

# Use pre-computed clean_text from CSV; fall back to live preprocessing if missing
if 'clean_text' in aug_df.columns:
    aug_df['clean_text'] = aug_df['clean_text'].fillna(aug_df['text'].apply(preprocess_text))
else:
    aug_df['clean_text'] = aug_df['text'].apply(preprocess_text)

aug_df['model_text'] = make_model_text(aug_df)

print(f'Augmented samples: {len(aug_df):,}')
print(aug_df['label'].value_counts().rename({0: 'No-PCL', 1: 'PCL'}))

# Concatenate and shuffle
train_df = pd.concat(
    [train_df, aug_df[['par_id', 'keyword', 'country', 'text', 'clean_text', 'model_text', 'label']]],
    ignore_index=True,
).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f'\nTrain (after augmentation): {len(train_df):,} | PCL={int((train_df.label==1).sum()):,} | No-PCL={int((train_df.label==0).sum()):,}')

# Class weights to handle any remaining imbalance
cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=train_df['label'].values)
print(f'Class weights -> No-PCL: {cw[0]:.4f},  PCL: {cw[1]:.4f}')

In [ ]:
# ============================================================
# Tokenisation
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def to_hf_dataset(frame: pd.DataFrame, has_labels: bool = True) -> Dataset:
    cols = ['model_text', 'label'] if has_labels else ['model_text']
    return Dataset.from_pandas(
        frame[cols].rename(columns={'model_text': 'text'}),
        preserve_index=False,
    )


def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)


def build_dataset(frame: pd.DataFrame, has_labels: bool = True) -> Dataset:
    ds = to_hf_dataset(frame, has_labels).map(tokenize, batched=True, remove_columns=['text'])
    if has_labels:
        ds = ds.rename_column('label', 'labels')
        ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
    else:
        ds.set_format(type='torch', columns=['input_ids', 'attention_mask'])
    return ds


train_ds = build_dataset(train_df)
dev_ds   = build_dataset(dev_df)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(train_ds)
print(dev_ds)

In [ ]:
# ============================================================
# Evaluation metrics (macro-F1 focused)
# ============================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'f1_macro':        f1_score(labels, preds, average='macro',  zero_division=0),
        'precision_macro': precision_score(labels, preds, average='macro', zero_division=0),
        'recall_macro':    recall_score(labels, preds, average='macro', zero_division=0),
    }

In [ ]:
# ============================================================
# Custom Trainer with class-weighted CrossEntropyLoss
# ============================================================
class WeightedCETrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = (
            torch.tensor(class_weights, dtype=torch.float)
            if class_weights is not None else None
        )

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        logits  = outputs.logits
        weight  = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss    = nn.CrossEntropyLoss(weight=weight)(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
# ============================================================
# Model + TrainingArguments + Trainer
# ============================================================
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

training_args = TrainingArguments(
    output_dir=RUN_NAME,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    optim='adamw_torch',           # AdamW from PyTorch
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,   # restore best checkpoint after training
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    save_total_limit=2,
    report_to='none',
    seed=SEED,
)

trainer = WeightedCETrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=cw.tolist(),
)

print('Trainer configured.')
print(f'Total training samples : {len(train_ds):,}')
print(f'Total dev samples      : {len(dev_ds):,}')

In [ ]:
train_result = trainer.train()
print(train_result)

In [ ]:
# ============================================================
# Evaluate best checkpoint on dev set
# ============================================================
eval_metrics = trainer.evaluate()
print('Dev metrics (best checkpoint):')
for k, v in eval_metrics.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

# Full classification report + confusion matrix
pred_out = trainer.predict(dev_ds)
y_true   = pred_out.label_ids
y_pred   = np.argmax(pred_out.predictions, axis=-1)

print('\n' + classification_report(y_true, y_pred, target_names=['No-PCL', 'PCL'], digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred No-PCL', 'Pred PCL'],
            yticklabels=['True No-PCL', 'True PCL'])
plt.title('Confusion Matrix — RoBERTa Best Model (Dev)')
plt.tight_layout()
plt.savefig(os.path.join(RUN_NAME, 'confusion_matrix.png'), dpi=150)
plt.show()

In [ ]:
# ============================================================
# Generate dev.txt  (predictions in official dev split order)
# ============================================================
# dev_df was already ordered to match dev_ids_df, so dev_ds rows align directly.
dev_preds = y_pred.tolist()   # reuse predictions from evaluate cell above

assert set(dev_preds).issubset({0, 1}), 'Unexpected label values in dev predictions'
assert len(dev_preds) == len(dev_df), (
    f'Dev count mismatch: {len(dev_preds)} predictions vs {len(dev_df)} rows'
)

with open(DEV_TXT_PATH, 'w') as f:
    f.write('\n'.join(str(p) for p in dev_preds) + '\n')

print(f'Written {len(dev_preds):,} predictions -> {DEV_TXT_PATH}')

In [ ]:
# ============================================================
# Generate test.txt  (3832 predictions in test file order)
# ============================================================
test_df = load_test_tsv(TEST_PATH)
test_df['clean_text'] = test_df['text'].apply(preprocess_text)
test_df['model_text'] = make_model_text(test_df)

test_ds       = build_dataset(test_df, has_labels=False)
test_pred_out = trainer.predict(test_ds)
test_preds    = np.argmax(test_pred_out.predictions, axis=-1).tolist()

assert set(test_preds).issubset({0, 1}), 'Unexpected label values in test predictions'
assert len(test_preds) == len(test_df), (
    f'Test count mismatch: {len(test_preds)} predictions vs {len(test_df)} rows'
)

with open(TEST_TXT_PATH, 'w') as f:
    f.write('\n'.join(str(p) for p in test_preds) + '\n')

print(f'Written {len(test_preds):,} predictions -> {TEST_TXT_PATH}  (expected 3832)')

In [ ]:
# ============================================================
# Sanity checks on output files
# ============================================================
for path in [DEV_TXT_PATH, TEST_TXT_PATH]:
    with open(path) as f:
        lines = [l.strip() for l in f if l.strip()]
    print(f'{path:12s}: lines={len(lines):,}, unique_labels={sorted(set(lines))}')